# Does a Pokémon classifier mistake real animals for Pokémon?

**Results of the preliminary out-of-distribution (OOD) analysis**

A Vision Transformer trained to recognise 1025 Pokémon is shown photos of real animals, which are not among its classes. We measure how confident it is on these images, which animals it confuses with which Pokémon, and whether removing the background or turning the photos into cartoon-like drawings changes its behaviour.

### Key findings

1. **On Pokémon the model is accurate and confident:** 97.9 % accuracy, median confidence 0.99. Its wrong predictions are much less confident than its correct ones.
2. **Real animals are mostly recognised as "unknown":** median confidence about 0.11, normalized entropy about 0.70–0.72. Still, **3–5 % of the animal photos are confident mistakes** (confidence > 70 %).
3. **The confident mistakes are not random:** bison/cattle → Bouffalant, antelope/deer → Sawsbuck, pigeon → Unfezant, zebra → Zebstrika. Silhouette and distinctive features (horns, antlers, stripes) appear to drive them.
4. **Removing the background hardly changes the confidence**, but it often changes *which* Pokémon is predicted.
5. **Cartoonizing makes the model slightly more decided, but not more confidently wrong:** slightly lower entropy, but fewer confident mistakes, spread over more Pokémon.

## 1. Problem formulation

Image classifiers always assign an input to one of their known classes, even if the input belongs to none of them. Ideally, the model signals this by a **low confidence** and a **high entropy** (probability spread over many classes). In practice, models are often *overconfident* on out-of-distribution inputs, which makes their predictions hard to trust.

A Pokémon classifier is an interesting test case, because many Pokémon are designed after real animals. Real animal photos are therefore OOD for the model, but visually related to some of its classes.

**Research questions**

- **RQ1:** How confident is the classifier on its own classes (Pokémon) compared with real animals?
- **RQ2:** Which animals does the model confidently confuse with which Pokémon, and what do these cases have in common?
- **RQ3:** Does the background of the animal photos influence the predictions?
- **RQ4:** Does the model become more confident if the animals look more like Pokémon artwork (cartoon style)?

## 2. Setup

### Model

[skshmjn/Pokemon-classifier-gen9-1025](https://huggingface.co/skshmjn/Pokemon-classifier-gen9-1025): a Vision Transformer (ViT) fine-tuned on 1025 Pokémon classes. It is used as is, without any further training.

### Datasets

| Dataset | Images | Classes | Description |
|---|---|---|---|
| Pokémon | 2485 | 893 | Classes the model was trained on (in-distribution), [Kaggle Pokémon image dataset](https://www.kaggle.com/datasets/hlrhegemony/pokemon-image-dataset) |
| Animals | 5400 | 90 | Real animal photos (OOD), [animal-matting](https://github.com/bruinxiong/animal-matting) |
| AM-2k original | 2000 | 20 | Real animal photos (OOD), AM-2k |
| AM-2k background removed | 2000 | 20 | The same photos, cut out with the dataset's masks and placed on a white background, like the Pokémon images |
| AM-2k cartoonized | 2000 | 20 | The background-removed photos turned into cartoon-like drawings (see 3.5) |

The three AM-2k variants contain the same 2000 photos, so they can be compared photo by photo.

### Measures

For each image the model outputs a probability distribution $p$ over the $K = 1025$ classes. We use:

- **Confidence:** the highest probability, $\max_k p_k$.
- **Normalized entropy:** $H(p) / \log K$, between 0 (all probability on one class) and 1 (probability spread evenly over all classes).
- **Confident prediction:** confidence > 70 %. Since none of the animals is a Pokémon, every confident prediction on an animal is a **confident mistake**.

## 3. Results

### 3.1 Confidence on Pokémon vs. real animals (RQ1)

| Dataset | Accuracy | Mean confidence | Median confidence | Mean norm. entropy | Confidence > 70 % | Confidence > 90 % |
|---|---|---|---|---|---|---|
| Pokémon | 97.9 % | 0.952 | 0.991 | 0.038 | 95.3 % | 89.0 % |
| Animals | – | 0.199 | 0.120 | 0.702 | 4.7 % (252) | 1.4 % (74) |
| AM-2k original | – | 0.181 | 0.107 | 0.721 | 3.7 % (74) | 1.3 % (25) |
| AM-2k background removed | – | 0.183 | 0.113 | 0.720 | 3.2 % (65) | 1.2 % (23) |

- On **Pokémon**, the model is almost always right and very sure. The 51 wrong predictions have a mean confidence of 0.59, compared with 0.96 for the correct ones, so low confidence is a useful warning sign.
- On **real animals**, the typical prediction has low confidence and high entropy: the model effectively signals "I don't know".
- However, **3–5 % of the animal photos receive a confidence above 70 %**, and about 1 % even above 90 %.

> 📊 **Figure: Confidence vs. normalized entropy** (scatter plot, one point per image, all four datasets)  
> *From `analysis.ipynb`, section "Confidence vs. normalized entropy".*  
> Pokémon cluster in the bottom-right corner (high confidence, low entropy), the animals mostly in the top-left. Animal points in the bottom-right corner are the confident mistakes.

### 3.2 Confident mistakes: which animals become which Pokémon? (RQ2)

Most frequent (animal → Pokémon) pairs among the predictions with confidence > 70 %:

| Dataset | Animal | Predicted Pokémon | Images | Mean confidence |
|---|---|---|---|---|
| Animals (252 confident, 61 different pairs) | bison | Bouffalant | 43 | 0.96 |
| | pigeon | Unfezant | 27 | 0.80 |
| | antelope | Sawsbuck | 25 | 0.82 |
| | ox | Bouffalant | 18 | 0.89 |
| | flamingo | Flamigo | 15 | 0.88 |
| AM-2k original (74 confident, 17 different pairs) | cattle | Bouffalant | 30 | 0.91 |
| | antelope | Sawsbuck | 17 | 0.79 |
| | cat | Mightyena | 9 | 0.80 |
| | zebra | Zebstrika | 4 | 0.80 |

The mistakes are **visually plausible**. The model maps an animal to a Pokémon that is designed after a similar animal (bison → Bouffalant, flamingo → Flamigo, pigeon → Unfezant), or that shares a distinctive feature such as horns, antlers or stripes (antelope/deer → Sawsbuck, zebra → Zebstrika). Silhouette and such features seem to matter more than texture or colour.

> 📊 **Figure: Most confident animal photos next to the predicted Pokémon**  
> *From `analysis.ipynb`, section "Examples" (animals the model is most confident about).*

> 📊 **Figure (optional): One example per (animal → Pokémon) pair**  
> *From `analysis.ipynb`, section "Confident predictions on animals".*

### 3.3 Does the background matter? (RQ3)

The Pokémon images have a plain white background, the animal photos a natural one. To test whether the background causes the mistakes, each AM-2k photo is compared with its background-removed version.

| | Photos |
|---|---|
| Confident (> 70 %) with background | 74 |
| Confident without background | 65 |
| Confident in both versions | 41 |
| Confident only with background | 33 |
| Confident only without background | 24 |
| Same predicted Pokémon in both versions | 47 % of all photos |

- Removing the background barely changes **how many** photos are confident (3.7 % → 3.2 %), and mean confidence and entropy stay almost identical.
- It does change **which** photos are confident, and more than half of all photos get a different predicted Pokémon.
- The dominant mistake becomes even more dominant: cattle → Bouffalant rises from 30 to 37 of the confident predictions.

**Conclusion:** the background influences *which* Pokémon is chosen, but it is not what makes the model confident.

> 📊 **Figure: Confusion matrix, AM-2k background removed** (true animal vs. predicted Pokémon, confidence > 70 %)  
> *From `analysis.ipynb`, section "AM-2k (background removed): animal class vs. predicted Pokémon".*

### 3.4 Cartoonizing: do Pokémon-like drawings increase the confidence? (RQ4)

#### Method

The background-removed AM-2k photos were turned into cartoon-like drawings with OpenCV (`src/preprocessing.py`):

1. Edge-preserving smoothing (bilateral filter) removes fur and texture detail.
2. k-means reduces each animal to 6 flat colours, and small colour blotches are removed (median filter).
3. Colours are made lighter and more saturated, like the Pokémon artwork.
4. Dark outlines are drawn around the animal and along its strongest inner edges (Canny edge detection).

The filter knows nothing about Pokémon, so any change in the predictions comes from the drawing **style**, not from Pokémon-specific content. The parameters were tuned by eye. The images look more like drawings, but not really like Pokémon.

> 📊 **Figure: Example photos before and after cartoonizing**  
> *E.g. a few images from `images/AM-2k-bg-removed/` next to the same images in `images/AM-2k_bg-removed-cartoonized/`.*

#### Results

| | AM-2k original | AM-2k background removed | AM-2k cartoonized |
|---|---|---|---|
| Mean confidence | 0.181 | 0.183 | 0.191 |
| Median confidence | 0.107 | 0.113 | 0.137 |
| Mean norm. entropy | 0.721 | 0.720 | **0.687** |
| Median norm. entropy | 0.773 | 0.770 | **0.728** |
| Confident (> 70 %) | 74 (3.7 %) | 65 (3.2 %) | **50 (2.5 %)** |
| Very confident (> 90 %) | 25 | 23 | **7** |
| Different Pokémon among the confident predictions | 11 | 8 | **16** |

**Photo-by-photo comparison** (background removed vs. cartoonized, 2000 pairs):

- The normalized entropy is lower for **1271 of 2000 photos (64 %)**, on average by 0.033 (sign test p < 10⁻³⁰). The effect is small, but consistent.
- The confidence is higher for 57 % of the photos, on average by only 0.008.
- Only **20 %** of the photos keep the same predicted Pokémon.

**Per animal**, the effect differs strongly:

- Lions, bears, leopards and tigers become clearly more decided (mean entropy −0.11).
- Cattle goes the other way (entropy +0.21, confidence −0.29): the confident match with Bouffalant mostly disappears (37 → 4 images).

**Confident mistakes after cartoonizing** (most frequent pairs): antelope → Sawsbuck (15), zebra → Zebstrika (4), cattle → Bouffalant (4), deer → Sawsbuck (4), elephant → Piloswine (3). The 50 confident predictions are spread over 24 different (animal → Pokémon) pairs; without cartoonizing, 65 confident predictions were concentrated on 9 pairs.

**Conclusion:** cartoonizing makes the model slightly more decided (lower entropy), but **not** more confidently wrong. The confident mistakes become fewer and less systematic.

> 📊 **Figure: Confusion matrix, AM-2k cartoonized** (confidence > 70 %)  
> *From `analysis.ipynb`, last cell (confusion matrix of the confident cartoonized predictions).* Best placed next to the background-removed confusion matrix from 3.3.

> 📊 **Figure: Most confident cartoonized images next to the predicted Pokémon**  
> *From `analysis.ipynb`, section "Cartoonized AM-2k dataset" (prediction cell).*

> 📊 **Figure (optional): Confidence vs. normalized entropy for the three AM-2k variants**  
> *From `analysis.ipynb`, interactive chart in the "Cartoonized AM-2k dataset" section.*

## 4. Discussion

- **RQ1:** The classifier is reliable on its own classes, and on real animals its low confidence and high entropy mostly reveal that the input is unknown. Confidence or entropy could therefore serve as a simple OOD detector for most images.
- **RQ2:** A small share of animal photos (3–5 %) still produces confident mistakes. They are systematic rather than random: the model follows the silhouette and distinctive features of the animal and maps it to a Pokémon designed after a similar animal. A threshold-based OOD detector would miss exactly these cases.
- **RQ3:** The background is not the cause of the confident mistakes. It mainly influences which Pokémon is chosen.
- **RQ4:** A cartoon style alone does not fool the model more. What makes it confident seems to be the **shape** of a Pokémon-like animal (horns, antlers, body outline), not the drawing style.

## 5. Limitations and next steps

**Limitations**

- The cartoon filter is hand-tuned and judged by eye only. Its images do not really look like Pokémon, so RQ4 is only answered for this particular style.
- The 70 % threshold for a "confident" prediction is a choice; other thresholds may shift the numbers.
- The Pokémon dataset only has images of 893 of the 1025 classes, so some predicted Pokémon cannot be shown next to the animal.
- AM-2k covers only 20 animal classes; some effects (e.g. cattle → Bouffalant) depend on single classes.

**Next steps**

- Compare with a pretrained, general-purpose cartoonization model (e.g. AnimeGANv2). A model trained on Pokémon images should be avoided, because it would paint Pokémon features into the animals.
- Test the role of shape directly, e.g. with silhouettes only, or by removing horns/antlers from the images.
- Evaluate how well confidence or entropy separate Pokémon from animals as an OOD detector (e.g. ROC curve / AUROC).